In [2]:
import polars as pl
import scanpy as sp
import anndata as ad
import pandas as pd

In [53]:
adata = ad.read_h5ad(r"C:\Users\Leticia\Documents\Letworkspace\Sweep-Harmonization\Meus_testes\Controle_qualidade\dataM\matrizFiltradaeNormalizadaM.h5ad")

In [54]:
adata.obs

,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,n_genes
AAAGATGCACGGTGTC-1,727,1043.0,0.0,0.000000,727
AAATGCCTCCAATGGT-1,4350,11487.0,75.0,0.652912,4350
AACCATGTCTGTACGA-1,4309,10647.0,321.0,3.014934,4309
AACCGCGTCCGCATAA-1,2819,5736.0,123.0,2.144351,2819
AACGTTGGTTCAGGCC-1,4052,10029.0,45.0,0.448699,4052
...,...,...,...,...,...
TTCTTAGGTGTGCGTC-48,3709,7935.0,263.0,3.314430,3709
TTGTAGGCACAGACTT-48,5032,13640.0,149.0,1.092375,5032
TTGTAGGTCTAGAGTC-48,5449,14447.0,320.0,2.214993,5449
TTTATGCTCAGAAATG-48,4058,8746.0,66.0,0.754631,4058


In [55]:
adata.shape

(47523, 32738)

In [56]:
df = pd.read_csv("C:\\Users\\Leticia\\Documents\\Letworkspace\\pipiline_hopifield\\dadosExternos\\metadados_celulasm.csv", index_col=0)

In [57]:
print(df.columns.tolist())
print(df.head(2))


['celltype', 'indiv', 'diag']
                   celltype  indiv         diag
samples                                        
AAACGGGAGATCCCGC.1       Ex      1  low-amyloid
AAATGCCTCCAATGGT.1       Ex      1  low-amyloid


In [58]:
df.index = df.index.str.replace('.', '-', regex=False)

# Verificar sobreposição
comum = adata.obs.index.intersection(df.index)
print(f"Células em comum: {len(comum)} de {adata.n_obs}")

# Vincular ao adata
for col in df.columns:
    adata.obs[col] = df[col].reindex(adata.obs.index).values


Células em comum: 45663 de 47523


In [59]:
df.index = df.index.str.replace('.', '-', regex=False)


In [60]:
df.head

<bound method NDFrame.head of                     celltype  indiv          diag
samples                                          
AAACGGGAGATCCCGC-1        Ex      1   low-amyloid
AAATGCCTCCAATGGT-1        Ex      1   low-amyloid
AACCATGTCAGTGCAT-1        Ex      1   low-amyloid
AACCATGTCTGTACGA-1        Ex      1   low-amyloid
AACCGCGTCCGCATAA-1        Ex      1   low-amyloid
...                      ...    ...           ...
AGGCCGTAGAGCAATT-48      Per     48  high-amyloid
AGTGAGGGTGCAACGA-48      Per     48  high-amyloid
CACACTCTCTCTGAGA-48      Per     48  high-amyloid
TAGTGGTAGAATGTTG-48      End     48  high-amyloid
TGCCCATAGTAGGTGC-48      Per     48  high-amyloid

[70634 rows x 3 columns]>

In [52]:
for col in df.columns:
    adata.obs[col] = df[col].reindex(adata.obs.index).values

print(adata.obs[['celltype', 'indiv', 'diag']].head())
print(adata.obs['celltype'].isna().sum(), "células sem anotação")
for col in df.columns:
    adata.obs[col] = df[col].reindex(adata.obs.index).values

print(adata.obs[['celltype', 'indiv', 'diag']].head())
print(adata.obs['celltype'].isna().sum(), "células sem anotação")


                   celltype  indiv         diag
AAAGATGCACGGTGTC-1      Oli    1.0  low-amyloid
AAATGCCTCCAATGGT-1       Ex    1.0  low-amyloid
AACCATGTCTGTACGA-1       Ex    1.0  low-amyloid
AACCGCGTCCGCATAA-1       Ex    1.0  low-amyloid
AACGTTGGTTCAGGCC-1       Ex    1.0  low-amyloid
1860 células sem anotação
                   celltype  indiv         diag
AAAGATGCACGGTGTC-1      Oli    1.0  low-amyloid
AAATGCCTCCAATGGT-1       Ex    1.0  low-amyloid
AACCATGTCTGTACGA-1       Ex    1.0  low-amyloid
AACCGCGTCCGCATAA-1       Ex    1.0  low-amyloid
AACGTTGGTTCAGGCC-1       Ex    1.0  low-amyloid
1860 células sem anotação


In [61]:

adata = adata[adata.obs['celltype'].notna()].copy()
print(f"Células restantes: {adata.n_obs}")


Células restantes: 45663


In [62]:
adata.write_h5ad(r"C:\Users\Leticia\Documents\Letworkspace\Sweep-Harmonization\Meus_testes\Controle_qualidade\dataM\matrizFiltradaeNormalizadaMParcial.h5ad")


In [67]:
adata.shape

(45663, 32738)

In [68]:
adata.obs[['celltype']].to_csv("outputs/celltypeM.csv")



In [69]:

adata = adata[adata.obs['celltype'].notna()].copy()
print(f"Células restantes: {adata.n_obs}")


Células restantes: 45663


In [71]:
adata.obs[['celltype']].to_csv("outputs/celltypeMparcial.csv")



In [26]:
adata.write_h5ad("outputs/adataM_binarizado_alinhado.h5ad")

In [82]:
df_cel = pd.read_csv(r"C:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\celltypeMparcial.csv", index_col=0)
print(df_cel.head(2))
df_cel.shape

                   celltype
AAAGATGCACGGTGTC-1      Oli
AAATGCCTCCAATGGT-1       Ex


(45663, 1)

In [83]:
df_cel = df_cel[['celltype']].reset_index(drop=True)
print(df_cel.head(2))

  celltype
0      Oli
1       Ex


In [84]:
df_cel.to_csv("outputs/celltypeMparcial.csv", index=False)

### 5.Adicionando tipo celular aoa dados fujita apartir do tipo celular binario para texto

In [19]:
import pandas as pd

celltypeBinF = pd.read_csv(r"C:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\dadosExternos\cell_types_binario.txt",header=None)
celltypeBinF.head(2)
celltypeBinF.replace



<bound method NDFrame.replace of        0
0      3
1      0
2      0
3      0
4      3
...   ..
40908  0
40909  0
40910  0
40911  0
40912  0

[40913 rows x 1 columns]>

In [21]:
celltypeBinF.head(2)

,0
0,3
1,0


In [22]:
mapeamento_inv = {v: k for k, v in mapeamento.items()}

df_novo = celltypeBinF.copy()
df_novo['cel_name'] = df_novo.iloc[:, 0].map(mapeamento_inv)

print(df_novo.head())


   0  cel_name
0  3       NaN
1  0       NaN
2  0       NaN
3  0       NaN
4  3       NaN


In [23]:
celltypeBinF = celltypeBinF.rename(columns={celltypeBinF.columns[0]: 'celltype_bin'})


In [25]:
mapeamento = {
    'Astrocyte': 1,
    'Endothelial': 2,
    'Excitatory Neurons': 3,
    'Inhibitory Neurons': 4,
    'Microglia': 5,
    'Oligodendrocytes': 6,
    'OPCs': 7,
    'Pericytes': 8,
    'Fibroblast': 9,
    'NFOL/MOL':10,
    'Macrophages':11,
    'Monocytes':12,
    'SMC':13,
    'COP':14,
    'NK Cells':15,
    'Neutrophils':16,
    'CD8+ T Cells':17,
    'Erythrocytes':18,
}

In [26]:
mapeamento = {v: k for k, v in mapeamento.items()}

df_novo = celltypeBinF.copy()
df_novo['cel_name'] = df_novo.iloc[:, 0].map(mapeamento)

print(df_novo.head())


   celltype_bin            cel_name
0             3  Excitatory Neurons
1             0                 NaN
2             0                 NaN
3             0                 NaN
4             3  Excitatory Neurons


#### salvo cel tipe bimn  mais o nome para fujita 

In [27]:
df_novo.to_csv("dadosExternos/celltypeBinF_eceltypename.csv", index=False)